# Inference from a diffusion_ds checkpoint

This notebook loads a trained `.pth` checkpoint, rebuilds the pipeline from the config, and generates one HR sample from LR and geo inputs.

In [ ]:
from pathlib import Path

import torch
from mmengine.config import Config
from mmengine.runner import Runner

repo_root = Path.cwd().resolve()
train_res_root = repo_root / "work_dirs/res_diffusion_ds"
data_prefix = repo_root / "data/downscaling/pt"

In [ ]:
config = train_res_root / "base.py"
checkpoint = train_res_root / "epoch_1.pth"

split_dir = "test"
filename = "000000.dat"

out_dir = train_res_root / "infer"
out_dir.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

For the synthetic dataset generated by `make_toy_downscaling_data.py`, valid file names are `000000.dat` and `000001.dat` for `train`, and `000000.dat` for `validation` and `test` with the default sample counts.

In [ ]:
cfg = Config.fromfile(str(config))
cfg.log_level = "ERROR"
runner = Runner.from_cfg(cfg)

In [ ]:
state_dict = torch.load(checkpoint, map_location="cpu")
if "state_dict" in state_dict:
    state_dict = state_dict["state_dict"]

_ = runner.model.load_state_dict(state_dict, strict=False)

In [ ]:
model = runner.model.to(device)
model.eval()
model.set_pipeline()
pipeline = model.pipeline.to(device)

In [ ]:
LR_images = {}
for name in pipeline.LR_list:
    path = data_prefix / split_dir / name / "LR" / filename
    LR_images[name] = torch.load(path, map_location="cpu").unsqueeze(0).to(device)
    print(name, tuple(LR_images[name].shape))

In [ ]:
geo_data = {}
for name in model.vae.geo_list or []:
    path = data_prefix / f"{name}.dat"
    tensor = torch.load(path, map_location="cpu")
    if tensor.ndim == 2:
        tensor = tensor.unsqueeze(0)
    geo_data[name] = tensor.unsqueeze(0).to(device)
    print(name, tuple(geo_data[name].shape))

In [ ]:
generator = torch.Generator(device=device).manual_seed(0)
with torch.no_grad():
    result = pipeline(
        LR_images=LR_images,
        geo_data=geo_data,
        num_inference_steps=50,
        generator=generator,
    )

In [ ]:
for name, tensor in result.items():
    save_path = out_dir / f"{name}_{Path(filename).stem}.pt"
    torch.save(tensor.cpu(), save_path)
    print(name, tuple(tensor.shape), "->", save_path)

In [ ]:
model.del_pipeline()